<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/Optional_ALM_Codegen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Asset Lifecycle Management - Code Generation & Complex Queries (Optional)

> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

> **Prerequisites:** none in Colab — the setup cell prepares everything; the sandbox and backend are started below.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand when code generation is needed** vs pre-built tools
2. **Set up a code execution sandbox** safely
3. **Configure the code generation assistant** in NAT
4. **Run complex queries** requiring custom data transformations
5. **Inspect generated code** for correctness
6. **Handle code execution errors** and debugging
7. **Understand security considerations** for code execution

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

In [ ]:
from nat_client import NATClient, link
client = NATClient()

> **Reset codebase:** If you need to restore all source files and config to their original state (e.g. after testing a solution), run the cell below.

In [ ]:
client.reset()

---
# Section O.1: Code Generation Overview

## When Do We Need Code Generation?

Pre-built tools are great for common operations, but some queries require **custom logic**:

### Examples:

**❌ Cannot use pre-built tools:**
```
"Calculate the moving average of sensor_measurement_3 with a 10-cycle window and compare it to sensor_measurement_5"
```
→ Needs: Custom pandas transformation

**❌ Cannot use pre-built tools:**
```
"Compute a dual-axis chart showing sensor degradation alongside capped RUL"
```
→ Needs: Custom pandas + matplotlib multi-axis plotting

### Code Generation Flow:

```
User Query
    ↓
Reasoning Agent identifies: "Need custom code"
    ↓
Calls code_generation_assistant with requirements
    ↓
Coding LLM generates Python code
    ↓
Code sent to execution sandbox
    ↓
Sandbox executes code in isolated environment
    ↓
Returns: Output data, visualizations, or error
    ↓
Agent incorporates results into response
```

## NAT's Code Execution Architecture

### Security Model:

- **Isolated Sandbox**: Code runs in separate container/process
- **Limited Filesystem Access**: Only specific directories mounted
- **No Network Access**: Cannot make external API calls
- **Resource Limits**: Memory and CPU constraints
- **Timeout Protection**: Max execution time enforced

### Components:

1. **Coding LLM**: Generates Python code (`nvidia/nemotron-3.5-lightning-30b-a3b` in this edition)
2. **Code Generation Assistant**: Orchestrates code creation and retry logic
3. **Code Execution Tool**: Sends code to the sandbox
4. **Local Sandbox**: in the lab, a Docker container sharing the `agent_workspace` volume; in Colab, `sandbox_server.py` — a small HTTP server that runs each snippet in a fresh Python subprocess inside `agent_workspace/` (same `/execute` API)

## Why This Matters

Code generation enables:
- **Flexibility**: Handle any custom analysis
- **Power**: Full Python/pandas capabilities
- **Adaptability**: No need to pre-build every tool
- **User-driven**: Responds to novel user requests

But requires:
- **Careful security**: Isolated execution
- **Error handling**: Generated code can fail
- **Validation**: Check code before execution
- **Monitoring**: Track what code is being generated

---
# Section O.2: Sandbox Setup

## Step 1: Start and Verify the Code Execution Sandbox

In the lab the sandbox is a **Docker Compose service** (`code_execution_sandbox`). Docker is not available in Colab, so this edition ships `sandbox_server.py`, which exposes the same API:

**How it works:**
- HTTP server on port 6000 accepting code via `POST /execute` (`generated_code`, `timeout`, `language`)
- Runs every request in a fresh Python subprocess with `agent_workspace/` as the working directory (the lab's `/workspace` mount; `/workspace` is symlinked to it)
- Database is reachable at `/workspace/database/nasa_turbo.db`, exactly like inside the lab sandbox
- Timeout protection per request

> ⚠️ It isolates crashes and timeouts but is **not a security boundary** — fine for a disposable Colab VM, not for production.

**Config URI:** `http://127.0.0.1:6000/execute`

In [ ]:
client.start_sandbox()

### Test Sandbox

In [ ]:
import requests
url = "http://127.0.0.1:6000/execute"

payload = {
    "generated_code": "print('Sandbox connection successful!')",
    "timeout": 30,
    "language": "python"
}
try:
    response = requests.post(url, json=payload)
    print(f"Status Code: {response.status_code}")
    print("Response Body:", response.json())
except Exception as e:
    print(f"Request failed: {e}")

## Step 2: Update NAT Configuration

The base config from previous notebooks does **not** include code generation. We need to add three things:

1. A **coding LLM** to generate Python code
2. A **code_generation_assistant** tool wired to the local sandbox
3. The tool added to `data_analysis_assistant.tool_names`


<details>
<summary><b>📋 Solution (Colab): apply the config edits programmatically</b></summary>

Run the cell below — `exercise_solutions.optional_codegen()` makes these three additions (idempotent).

> **To restore original files after testing:** run `client.reset()` — this reverts all source files and config back to the base state.
</details>

<details>
<summary><b>Click to see what changed in the config</b></summary>

**1. New LLM** added under `llms:`:
```yaml
  coding_llm:
    _type: nim
    model_name: "nvidia/nemotron-3.5-lightning-30b-a3b"
    max_tokens: 16384
```

**2. New functions** added under `functions:`:
```yaml
  code_execution:
    _type: code_execution
    uri: http://127.0.0.1:6000/execute
    sandbox_type: "local"
    timeout: 60
    max_output_characters: 2000

  code_generation_assistant:
    _type: code_generation_assistant
    llm_name: "coding_llm"
    code_execution_tool: "code_execution"
    output_folder: "agent_workspace"
    verbose: true
    max_retries: 1
```

**3. Tool registered** in `data_analysis_assistant.tool_names`:
```yaml
    tool_names: [
      "sql_retriever", "predict_rul",
      "plot_distribution", "plot_line_chart", "plot_comparison",
      "anomaly_detection", "plot_anomaly",
      "code_generation_assistant"   # new
    ]
```
</details>

## Step 3: Restart NAT Backend

After applying the config, restart your backend:

In [ ]:
import exercise_solutions as sol
sol.optional_codegen()

In [ ]:
client.restart()

In [ ]:
client.check_health()

---
# Section O.3: Complex Query Testing

## Test Query 1: Moving Average Analysis

**Run this using the client below:**

```
For unit 50 in FD001, calculate the 5-cycle moving average of sensor_measurement_2 and plot it alongside the original values
```

In [ ]:
client.query("For unit 50 in FD001, calculate the 5-cycle moving average of sensor_measurement_2 and plot it alongside the original values",
             timeout=900, show_steps=True)

In [ ]:
# Show the most recent image the agent produced
print("latest files:", client.latest_files(6))
client.show("*.png", latest=1)

### Expected Output:

- A line chart showing raw sensor_measurement_2 values and a smoothed 5-cycle moving average overlay
- The moving average should help reveal the underlying trend by filtering out cycle-to-cycle noise

## Test Query 2: Sensor Degradation with Capped RUL (Dual-Axis Chart)

This query retrieves sensor data, then uses code generation to compute RUL and create a dual-axis visualization showing sensor degradation alongside remaining useful life.

**Run this with intermediate steps enabled (cell below):**

```
Retrieve time_in_cycles for unit 1 in train_FD001, ordered by time_in_cycles. Then use the code generation assistant to read the JSON file, calculate True RUL as (max_cycle - current_cycle) and Piece-wise RUL as min(True RUL, 125). Plot both True RUL and Piece-wise RUL against time_in_cycles on the same chart. Save as piecewise_rul.png
```

### What to Observe:

1. **Reasoning Agent** creates a multi-step plan
2. **Data Analysis Agent** calls:
   - SQL retrieval tool — fetches sensor_measurement_7 time series (saved as JSON)
   - **Code generation assistant** — reads the JSON, computes RUL and capped RUL, generates a dual-axis plot
3. **Generated Code** appears in intermediate steps
4. **Execution Output** shows the saved plot path

### Expected Output:

- A dual-axis chart with sensor_measurement_7 on the left y-axis (showing degradation over time) and capped RUL on the right y-axis (decreasing to 0 at failure, capped at 125)
- sensor_measurement_7 for unit 1 in FD001 shows clear HPC outlet pressure degradation

In [ ]:
client.query("Retrieve time_in_cycles for unit 1 in train_FD001, ordered by time_in_cycles. Then use the code generation assistant to read the JSON file, calculate True RUL as (max_cycle - current_cycle) and Piece-wise RUL as min(True RUL, 125). Plot both True RUL and Piece-wise RUL against time_in_cycles on the same chart. Save as piecewise_rul.png",
             timeout=900, show_steps=True)

In [ ]:
client.show("piecewise_rul.png")

---
# Section O.4: Exercise O.1 - Custom Code-Gen Query

## Objective

Create and test your own complex query requiring code generation.

## Deliverables

For your custom query, document:

1. **Query text**: What you asked
2. **Generated code**: Copy the Python code from intermediate steps
3. **Output**: Describe/screenshot the result
4. **Code quality**: Was the generated code correct? Any errors?
5. **Improvements**: What could be better?

## Common Issues & Solutions

### Issue: Code execution timeout
**Solution:** Break complex queries into simpler sub-queries

### Issue: File path errors
**Solution:** Code runs inside `agent_workspace/`. Refer to files by name only (e.g. `data.json`)

### Issue: Missing library
**Solution:** Check sandbox has required packages (pandas, matplotlib, numpy, seaborn)

### Issue: Generated code has syntax errors
**Solution:** The coding LLM isn't perfect. Try rephrasing your query with more detail

In [ ]:
# Exercise O.1 — your own code-generation query (edit me)
client.query("For unit 30 in FD001 training data, compute the rolling standard deviation (window 10) of sensor_measurement_11 "
             "using the code generation assistant and plot it against time_in_cycles. Save as rolling_std_unit30.png",
             timeout=900, show_steps=True)
client.show("rolling_std_unit30.png")

---
# Key Takeaways

In this notebook, you learned:

1. ✅ When code generation is needed vs pre-built tools
2. ✅ How to set up a secure code execution sandbox
3. ✅ How to configure code generation in NAT
4. ✅ How to run complex queries requiring custom code
5. ✅ How to inspect and validate generated code
6. ✅ Security considerations for production deployment
7. ✅ Trade-offs between flexibility and security

## Code Generation Best Practices

- **Be specific**: Detailed queries generate better code
- **Validate outputs**: Check generated visualizations
- **Monitor closely**: Log all generated code
- **Secure sandbox**: Never run untrusted code directly
- **Test thoroughly**: Try edge cases
- **Have fallbacks**: Pre-built tools as backup

## What's Next?

You've completed the optional Code Generation notebook. Continue to the Assessment to earn your certificate!

**[Continue to Final Assessment →](ALM_assessment.ipynb)**